# ZS601 2DGS LiDAR B → C on L4\nRuns B first, then C only after B succeeds. Full resolution; fixed 10-view previews; 150k iterations each.\n

In [ ]:
# ZS601 LiDAR B, then C on one L4 runtime
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys, json, time

os.environ['TORCH_CUDA_ARCH_LIST'] = '8.9'
subprocess.run(['nvidia-smi'], check=True)
print('Python:', sys.version, flush=True)

repo = Path('/content/ZS601_3DGS')
if not repo.exists():
    subprocess.run([
        'git','clone','--recursive','--branch','2dgs-zs601-mask-init',
        'https://github.com/VISjudy/ZS601_3DGS.git',str(repo)
    ], check=True)
source = repo/'2d-gaussian-splattingWithMask'
launcher = source/'colab/run_parallel_experiment.py'
subprocess.run([sys.executable,'-m','py_compile',str(launcher)], check=True)

print('SERIAL PLAN: B (LiDAR, distortion off) -> C (LiDAR, distortion=1000)', flush=True)
for mode in ('b','c'):
    print(f'===== START {mode.upper()} =====', flush=True)
    rc = subprocess.call([sys.executable,'-u',str(launcher),'--mode',mode], cwd=str(source))
    print(f'{mode.upper()}_RETURN_CODE={rc}', flush=True)
    if rc != 0:
        raise RuntimeError(f'{mode.upper()} failed. Later stages are intentionally paused.')
print('B AND C COMPLETE', flush=True)